In [1]:
import random
import string
import nltk
from nltk.corpus import movie_reviews, stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [4]:
for resource in ["movie_reviews", "punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(resource, quiet=True)

In [5]:
documents = [
    (movie_reviews.raw(fileid), category)
    for category in movie_reviews.categories()
    for fileid in movie_reviews.fileids(category)
]

random.seed(42)
random.shuffle(documents)

texts = [doc for doc, label in documents]
labels = [label for doc, label in documents]
print(f"Total reviews loaded: {len(texts)}")
print(f"Class distribution: {labels.count('pos')} pos, {labels.count('neg')} neg\n")

Total reviews loaded: 2000
Class distribution: 1000 pos, 1000 neg



In [19]:
# texts

In [20]:
# labels

### Text preprocessing: tokenization, lowercasing, stopword removal,punctuation removal, lemmatization

In [9]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()
punctuation_table = str.maketrans("", "", string.punctuation)


def preprocess(text: str) -> str:
    text = text.lower()
    text = text.translate(punctuation_table)          # remove punctuation
    tokens = word_tokenize(text)                       # tokenization
    tokens = [t for t in tokens if t.isalpha()]         # keep only alphabetic tokens
    tokens = [t for t in tokens if t not in stop_words] # remove stopwords
    tokens = [lemmatizer.lemmatize(t) for t in tokens]  # lemmatization
    return " ".join(tokens)



In [22]:
print("Preprocessing reviews .....")
print()
cleaned_texts = [preprocess(t) for t in texts]
print("Sample before:", texts[0][:150].replace("\n", " "), "...")
print("=="*50)
print("Sample after :", cleaned_texts[0][:150], "...\n")

Preprocessing reviews .....

Sample before: mr . bean , a bumbling security guard from england is sent to la to help with the grandiose homecoming of a masterpiece american painting .  the first ...
Sample after : mr bean bumbling security guard england sent la help grandiose homecoming masterpiece american painting first two word said enough let know occurs bea ...



In [11]:
# 3. Train/test split

X_train_text, X_test_text, y_train, y_test = train_test_split(
    cleaned_texts, labels, test_size=0.2, random_state=42, stratify=labels
)

## Bag-of-Words and TF-IDF

In [12]:
bow_vectorizer = CountVectorizer(max_features=5000)
X_train_bow = bow_vectorizer.fit_transform(X_train_text)
X_test_bow = bow_vectorizer.transform(X_test_text)

tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_text)
X_test_tfidf = tfidf_vectorizer.transform(X_test_text)

# Multinomial Naive Bayes and Logistic Regression

In [13]:
def train_and_evaluate(model, X_train, X_test, y_train, y_test, name):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    print(f"--- {name} ---")
    print(f"Accuracy: {acc:.4f}")
    print(classification_report(y_test, preds))
    print("Confusion Matrix:\n", confusion_matrix(y_test, preds, labels=["pos", "neg"]))
    print()
    return acc


In [14]:
results = {}

results["Naive Bayes + BoW"] = train_and_evaluate(
    MultinomialNB(), X_train_bow, X_test_bow, y_train, y_test, "Naive Bayes + BoW"
)

results["Naive Bayes + TF-IDF"] = train_and_evaluate(
    MultinomialNB(), X_train_tfidf, X_test_tfidf, y_train, y_test, "Naive Bayes + TF-IDF"
)

results["Logistic Regression + BoW"] = train_and_evaluate(
    LogisticRegression(max_iter=1000), X_train_bow, X_test_bow, y_train, y_test,
    "Logistic Regression + BoW"
)

results["Logistic Regression + TF-IDF"] = train_and_evaluate(
    LogisticRegression(max_iter=1000), X_train_tfidf, X_test_tfidf, y_train, y_test,
    "Logistic Regression + TF-IDF"
)


--- Naive Bayes + BoW ---
Accuracy: 0.7900
              precision    recall  f1-score   support

         neg       0.77      0.82      0.80       200
         pos       0.81      0.76      0.78       200

    accuracy                           0.79       400
   macro avg       0.79      0.79      0.79       400
weighted avg       0.79      0.79      0.79       400

Confusion Matrix:
 [[151  49]
 [ 35 165]]

--- Naive Bayes + TF-IDF ---
Accuracy: 0.7950
              precision    recall  f1-score   support

         neg       0.76      0.85      0.81       200
         pos       0.84      0.73      0.78       200

    accuracy                           0.80       400
   macro avg       0.80      0.79      0.79       400
weighted avg       0.80      0.80      0.79       400

Confusion Matrix:
 [[147  53]
 [ 29 171]]

--- Logistic Regression + BoW ---
Accuracy: 0.8300
              precision    recall  f1-score   support

         neg       0.83      0.83      0.83       200
         po

In [15]:
print("=" * 45)
print("SUMMARY OF RESULTS")
print("=" * 45)
for name, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{name:<32} {acc:.4f}")

SUMMARY OF RESULTS
Logistic Regression + BoW        0.8300
Logistic Regression + TF-IDF     0.8275
Naive Bayes + TF-IDF             0.7950
Naive Bayes + BoW                0.7900


# Prediction

In [18]:
best_model = LogisticRegression(max_iter=1000)
best_model.fit(X_train_tfidf, y_train)

sample_reviews = [
    "This movie was absolutely fantastic, the acting was superb!",
    "Worst film I have ever seen, a complete waste of time.",
    "It was okay, not great but not terrible either.",
]

print("\n" + "=" * 95)
print("PREDICTIONS ON CUSTOM REVIEWS New Data (Logistic Regression + TF-IDF)")
print("=" * 95)
for review in sample_reviews:
    cleaned = preprocess(review)
    vec = tfidf_vectorizer.transform([cleaned])
    pred = best_model.predict(vec)[0]
    print(f"Review : {review}\nPredicted sentiment: {pred}\n")



PREDICTIONS ON CUSTOM REVIEWS New Data (Logistic Regression + TF-IDF)
Review : This movie was absolutely fantastic, the acting was superb!
Predicted sentiment: pos

Review : Worst film I have ever seen, a complete waste of time.
Predicted sentiment: neg

Review : It was okay, not great but not terrible either.
Predicted sentiment: neg

